# Lab 2: Agents in Azure AI Foundry (15 min)

## Objectives
- Create an agent with the Azure OpenAI **Responses API**
- Use function calling (tools)
- Use the Code Interpreter
- Understand the cycle: Request → Tool Calls → Tool Outputs → Response

## Concepts

### What is an Agent?
An **Agent** in Foundry is an intelligent assistant that can:
- Use **tools** (code interpreter, file search, functions)
- Maintain **context** throughout a conversation
- **Reason** about which tool to use and when

### Lifecycle (Responses API)
```
1. Create AzureOpenAI client (with endpoint and key)
2. Send request with instructions and tools
3. Process response (may include tool calls)
4. Send tool results back
5. Get final response
```

### SDK
```python
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=FOUNDRY_ENDPOINT,
    api_key=FOUNDRY_KEY,
    api_version="2025-03-01-preview",
)
```

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from openai import AzureOpenAI

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=key,
    api_version="2025-03-01-preview",
)

print(f"✅ Client connected: {endpoint}")

✅ Cliente conectado: https://foundry-mod2.cognitiveservices.azure.com/


## 🖥️ Exercise 2.1: Simple Agent

Let's create a basic agent without tools — just with instructions.

In [ ]:
# Exercise 2.1: Chat with a simple agent via Responses API
response = client.responses.create(
    model=model,
    instructions="You are an Azure specialist. Respond concisely with practical examples.",
    input="Explain the difference between Azure Functions and Container Apps in 3 lines.",
)

print(f"✅ Response received (id: {response.id})")
print(f"   Status: {response.status}\n")

# Show response
for item in response.output:
    if item.type == "message":
        for c in item.content:
            if hasattr(c, "text"):
                print(f"🤖 {c.text}")

✅ Resposta recebida (id: resp_0606076e7cc7004b0069dd0a83881c8194bf39fd52a6c2c96c)
   Status: completed

🤖 Azure Functions é um serviço serverless orientado a eventos que executa pedaços de código (funções) de forma altamente escalável, ideal para tarefas pequenas e reactivas, como processar mensagens de uma fila.  
Azure Container Apps, por outro lado, permite executar aplicações em contentores, com maior controlo sobre escalabilidade, recursos e integração, sendo adequado para workloads mais complexos e persistentes.  
Exemplo: use Functions para processar triggers de Blob Storage e Container Apps para alojar microserviços em contentores.


## 🖥️ Exercise 2.2: Agent with Function Calling

The real power of agents is using **tools**. Let's create an agent that can look up information.

In [ ]:
# Exercise 2.2: Agent with function tools
import json

# Define functions the agent can call
def get_service_price(service: str) -> str:
    """Returns the estimated price of an Azure service."""
    prices = {
        "app service basic": "€50/month",
        "container apps": "€0.000012/vCPU-s",
        "functions consumption": "€0.000016/GB-s",
        "aks": "Free (control plane) + node costs",
        "cosmos db": "Starting at €25/month (serverless)",
    }
    return json.dumps({"service": service, "price": prices.get(service.lower(), "Price not available. Check azure.com/pricing")})

def get_recommended_region(use_case: str) -> str:
    """Recommends the best Azure region for a use case."""
    regions = {
        "europe": "West Europe (Netherlands) or North Europe (Ireland)",
        "portugal": "West Europe (closest to Portugal)",
        "global": "Use Traffic Manager with multiple regions",
        "ai": "East US or Sweden Central (best model availability)",
    }
    return json.dumps({"use_case": use_case, "recommendation": regions.get(use_case.lower(), regions["europe"])})

# Map of available functions
available_functions = {
    "get_service_price": get_service_price,
    "get_recommended_region": get_recommended_region,
}

# Register the functions as tools for the Responses API
tools = [
    {
        "type": "function",
        "name": "get_service_price",
        "description": "Gets the estimated price of an Azure service",
        "parameters": {
            "type": "object",
            "properties": {
                "service": {"type": "string", "description": "Azure service name (e.g.: app service basic, container apps)"}
            },
            "required": ["service"]
        }
    },
    {
        "type": "function",
        "name": "get_recommended_region",
        "description": "Recommends the best Azure region for a use case",
        "parameters": {
            "type": "object",
            "properties": {
                "use_case": {"type": "string", "description": "Use case (europe, portugal, global, ai)"}
            },
            "required": ["use_case"]
        }
    },
]

print("✅ Functions defined: get_service_price, get_recommended_region")

✅ Funções definidas: obter_preco_servico, obter_regiao_recomendada


In [ ]:
# Create agent with tools and test
instructions = "You are an Azure consultant. Use the available tools to provide accurate information about prices and regions."

# 1. Send initial request
response = client.responses.create(
    model=model,
    instructions=instructions,
    input="I want to create an app with Container Apps. How much does it cost and what's the best region for Portugal?",
    tools=tools,
)

print(f"📨 Initial response (status: {response.status})")

# 2. Process tool calls and send results
tool_outputs = []
for item in response.output:
    if item.type == "function_call":
        func = available_functions[item.name]
        args = json.loads(item.arguments)
        result = func(**args)
        print(f"   🔧 Tool call: {item.name}({args}) → {result}")
        tool_outputs.append({"type": "function_call_output", "call_id": item.call_id, "output": result})

# 3. Send tool results back to the model
if tool_outputs:
    response2 = client.responses.create(
        model=model,
        instructions=instructions,
        input=tool_outputs,
        tools=tools,
        previous_response_id=response.id,
    )
    print(f"\n📨 Final response (status: {response2.status})\n")
    for item in response2.output:
        if item.type == "message":
            for c in item.content:
                if hasattr(c, "text"):
                    print(f"🤖 {c.text}")

📨 Resposta inicial (status: completed)
   🔧 Tool call: obter_preco_servico({'servico': 'container apps'}) → {"servico": "container apps", "preco": "\u20ac0.000012/vCPU-s"}
   🔧 Tool call: obter_regiao_recomendada({'caso_uso': 'portugal'}) → {"caso_uso": "portugal", "recomendacao": "West Europe (mais pr\u00f3ximo de Portugal)"}

📨 Resposta final (status: completed)

🤖 Criar uma aplicação com Azure Container Apps tem um custo estimado de **€0.000012 por segundo de vCPU**. Este valor pode variar dependendo do tempo de execução e das especificações do serviço.

Quanto à região ideal, a **West Europe** é a melhor escolha, pois é a mais próxima de Portugal, garantindo menores latências e melhor desempenho.


## 🖥️ Exercise 2.3: Code Interpreter

The **Code Interpreter** allows the agent to write and execute Python code.

In [ ]:
# Exercise 2.3: Agent with Code Interpreter
response_code = client.responses.create(
    model=model,
    instructions="You are a data analyst. Use the code interpreter to perform calculations and create visualizations.",
    input="Calculate the Fibonacci sequence up to the 10th number and show the result in a formatted table.",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
)

print(f"✅ Response received (status: {response_code.status})\n")
for item in response_code.output:
    if item.type == "code_interpreter_call":
        print(f"💻 Code executed:\n{item.code}\n")
    elif item.type == "message":
        for c in item.content:
            if hasattr(c, "text"):
                print(f"🤖 {c.text}")

✅ Resposta recebida (status: completed)

💻 Código executado:
import pandas as pd

# Função para calcular a sequência de Fibonacci até ao nth número
def fibonacci_sequence(n):
    sequence = [0, 1]
    for i in range(2, n):
        sequence.append(sequence[i-1] + sequence[i-2])
    return sequence[:n]

# Calculando a sequência de Fibonacci até ao 10º número
fibonacci_numbers = fibonacci_sequence(10)

# Criar uma tabela formatada com os resultados
df_fibonacci = pd.DataFrame({
    'Ordem': range(1, 11),
    'Número de Fibonacci': fibonacci_numbers
})

df_fibonacci

🤖 Aqui está a tabela formatada com os números da sequência de Fibonacci até ao 10º número:

| Ordem | Número de Fibonacci |
|-------|----------------------|
|   1   |          0          |
|   2   |          1          |
|   3   |          1          |
|   4   |          2          |
|   5   |          3          |
|   6   |          5          |
|   7   |          8          |
|   8   |         13          |
|   9   |        

In [ ]:
# Summary of responses created
print("📋 Response IDs created in this session:")
print(f"   2.1 Simple agent:       {response.id[:30]}...")
print(f"   2.3 Code interpreter:   {response_code.id[:30]}...")
print("\n✅ Session complete! No persistent resources to clean up.")

## ✅ Summary

In this lab you learned:
- How to use the Azure OpenAI **Responses API** to create agents
- Creating simple agents with instructions
- Using **function calling** to access external data
- Using the **Code Interpreter** for computation

**Next:** [Lab 2.1 - Agents with Agent Service](lab02.1-agentes.ipynb)